# 🎯 Architecture: Apache Spark (The Execution Model)

**Advanced Big Data Concepts for Interview Preparation**

- **Created:** 2026-02-28
- **Focus:** Driver vs Executors, RDDs, Transformations vs Actions, Under the Hood
- **Tags:** `spark` | `architecture` | `big-data` | `citi-prep`

## 📖 The Core Concept in Plain Language

Apache Spark is a distributed computing engine. When a dataset is too massive to fit on a single laptop's RAM (e.g., a 10TB CSV), you must split the file into chunks and process those chunks simultaneously across many different computers.

Spark manages the "brain" (Driver) that tells all the "workers" (Executors) exactly what piece of the data to calculate.

### Why This Matters

- **In-Memory Speed:** Unlike Hadoop MapReduce, which writes intermediate steps to the hard drive, Spark caches data in RAM, making it 100x faster.
- **Horizontal Scaling:** Need to process 2x more data? Just tell AWS to add more Executor nodes. You don't have to rewrite your code.
- **Fault Tolerance:** If a worker node explodes mid-calculation, Spark knows exactly how to recreate its specific chunk of data on a new node without restarting the entire job.

### The Key Insight

Spark uses "Lazy Evaluation". It does absolutely no calculations until you desperately ask it for a final answer (an "Action").

## 🔍 Architecture Overview

In [ ]:
print("== Driver Program (The Boss) ==")
print("Holds the SparkContext. Defines the main function. Creates the DAG (plan of attack).")
print("\n== Cluster Manager (HR) ==")
print("YARN, Kubernetes, or Standalone. Decides which machines get to be workers.")
print("\n== Executors (The Workers) ==")
print("Nodes that actually crunch the CPU instructions and hold chunks of data in their RAM.")

## 🏗️ The Anatomy of Transformations vs Actions

A Spark script is just building a recipe (DAG - Directed Acyclic Graph). It doesn't bake the cake until you call an Action.

### Transformations (Lazy - Recipe Building)
`map()`, `filter()`, `groupby()`, `join()` -> These return a new DataFrame/RDD pointer, but run 0 seconds of CPU calculation.

### Actions (Eager - Oven Baking)
`count()`, `show()`, `collect()`, `write()` -> These trigger the Driver to execute the entire DAG across the executors.

In [ ]:
# Pseudo-code

# 1. Read a 5TB file (Transformation - lazily registered)
# df = spark.read.csv("s3://bucket/logs/")

# 2. Filter (Transformation - recipe updated)
# df_filtered = df.filter(df['status'] == 500)

# 3. Write (ACTION - Cluster goes brrrrr... CPU starts spinning)
# df_filtered.write.parquet("s3://bucket/out/")

## 🔄 The Danger of Shuffling

When a worker is calculating `filter(status=500)`, it only needs the data physically present on its own hard drive. This is called a **Narrow Dependency**.

But what if you call `groupby(employee_id)`? Employee 123's logs might be spread across Worker A, Worker B, and Worker Z. They must send data over the network to a single worker to aggregate it. This network movement is called a **Shuffle**.

In [ ]:
print("Narrow Dependencies: map, filter, union (Fast, no network I/O)")
print("Wide Dependencies: groupby, join, orderby !SHUFFLE! (Slow, heavy network traffic)")
print("\nInterview Tip: 'Optimizing Spark usually means minimizing the amount of data shuffled over the network.'")

## 🌍 Real-World Example: Out of Memory (OOM) via Collect

**The Citi Context:** A junior analyst wrote a Spark job that crashed the main cluster daily. Their job essentially said: 'Go out to 50 executors, filter down this 20GB subset of data, and `collect()` it back to print to the screen.'

### The Problem with `collect()`
`collect()` is an Action that takes the data distributed across the 50 workers and attempts to haul it all back into the single memory space of the Driver node.

The Driver only had 4GB of RAM allocated. Attempting to ingest 20GB instantly triggered a Java `OutOfMemoryError`.

In [ ]:
print("Solution:\nInstead of `df.collect()`, always use `df.write.parquet(s3_path)` to instruct the workers to dump their chunk directly to S3.")
print("Never pull Big Data back to the single Driver node unless it resolves to a tiny aggregated scalar metric like a single `count()` line.")

## 🎭 Advanced Memory Tuning: Repartition vs Coalesce

In [ ]:
print("If you have 10,000 tiny partitions, the Spark scheduler chokes on overhead.")
print("If you want to reduce partitions from 10k to 100:\n")
print("- `df.repartition(100)` -> Causes a full data Shuffle (bad).")
print("- `df.coalesce(100)` -> Does not shuffle data. It just logically combines existing partitions on the same node (Good!)")

## 🎤 5 Interview Q&A

### Q1: What is an RDD vs a DataFrame?
**Answer:** Resilient Distributed Dataset (RDD) is the low-level fundamental data structure of Spark; it is an abstraction of a distributed collection of objects. A DataFrame is a higher-level abstraction built on top of RDDs, but structured with rows and columns (like SQL), allowing Spark's Catalyst Optimizer to automatically optimize the execution plan.

---

### Q2: Why is Spark faster than Hadoop MapReduce?
**Answer:** MapReduce reads from HDFS, processes, and writes back to HDFS between *every single step*. Spark processes data directly in RAM across the executors, passing intermediate results in-memory, mitigating massive disk I/O bottlenecks.

---

### Q3: What is the Catalyst Optimizer?
**Answer:** It is Spark SQL's brain. If you write your code in a dumb way (e.g., Joining two huge tables, and *then* filtering by `date = 'today'`), Catalyst rewrites the DAG under the hood to push the filter up. It filters the tables first *before* joining them, saving immense compute time.

---

### Q4: Explain Broadcast Joins.
**Answer:** If you join a massive 5TB facts table with a tiny 1MB lookup dimension table, a normal Join causes a massive network Shuffle. A Broadcast Join copies that tiny 1MB table to the RAM of *every single Executor* first. Then, the workers can join the data locally (Narrow Dependency), entirely eliminating the Shuffle.

---

### Q5: How do you handle Data Skew?
**Answer:** Data Skew happens when one partition is 100x larger than the others because a grouping key is imbalanced (e.g., grouping traffic by 'country', and US traffic is 90%). It makes 1 worker process 90% of the job while the other 99 idle. I fix it using **Salting**: appending a random integer (0-10) to the US key, so Spark shards the US data across 10 workers, aggregates locally, and then rolls it up.

## 📚 Key Terminology to Master

| Term | Definition | Example |
|------|-----------|----------|
| **Driver** | The master node orchestrating the job execution plan | Creates SparkContext |
| **Executor** | Worker processes that run the tasks and hold data in RAM | Reads partitions from S3 |
| **DAG** | Directed Acyclic Graph - the recipe of transformations | Spark's internal plan |
| **Shuffle** | Redistributing data across the cluster over the network | Triggered by `join` or `groupby` |
| **Partition** | A logical chunk of the dataset handled by one core at a time | 1 part = 1 task |

## 💼 The Citi Narrative: Caching Repeated Traversals

### The Problem
An ETL pipeline validating Trade messages took 4 hours to run. The code read from S3, applied filters, and then printed counts. Later in the same script, it queried that identical filtered dataframe again for validation counts. Spark was lazily evaluating, meaning it was literally re-reading from S3 and re-calculating the filters *twice* from scratch.

### The Solution
I added a single line of code: `df_filtered.cache()` (or `persist()`), right before the first action.

### The Impact
When the first action hit, Spark read from S3, filtered, and then stored the results cleanly in the Executors' RAM. The second action hit, and instead of restarting from S3, it instantly read straight from RAM. Runtime was cut from 4 hours to 45 minutes.

## 💪 Practice Exercises

In [ ]:
# EXERCISE 1: ID the Actions
# Which of these triggers execution? map, select, count, saveAsTable
print("Action triggers: count, saveAsTable")

In [ ]:
# EXERCISE 2: Memory settings
# If your executor has 16GB of RAM, and you try to cache a 30GB dataframe.
# What happens?
print("Spark doesn't crash here. By default, it stores what it can in RAM, and 'spills' the rest to disk (which slows down reads, but prevents an OOM). Explicitly, this is MEMORY_AND_DISK mode.")

In [ ]:
# EXERCISE 3: Salting Concept
# You group by 'CompanyId'. Apple (ID=1) is 90% of the dataset causing Skew.
# How do you temporarily change the key to distribute Apple's load?
print("Create a new column: 'CompanyId_Random'. For Apple rows, make it Apple_1, Apple_2,... Apple_10 using a random integer. Group by that key first to parallelize safely.")

## 🎯 Summary: Key Takeaways

### What You've Learned
1. ✅ **Master/Slave:** Driver plans, Executors do.
2. ✅ **Lazy Eval:** Nothing happens without an Action (count, write, collect).
3. ✅ **Avoid Shuffle:** Joins/GroupBys are expensive network hogs.
4. ✅ **Broadcast Joins:** The magic trick to avoid Shuffle when joining a massive truth table to a small lookup table.
5. ✅ **Caching:** Always `.cache()` DataFrames if you plan to call Actions on them more than once.

### Interview Confidence Checklist
- [ ] Can explain how the Driver talks to Executors via tasks/partitions.
- [ ] Can name 3 Transformations and 3 Actions.
- [ ] Predict why a `.collect()` on a giant dataset will blow up the Driver.
- [ ] Explain the concept of Data Skew and Salting.
- [ ] Use the Citi story to explain the value of caching in memory.

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Understanding the engine under the hood separates standard Spark Coders from true Big Data Engineers. 🚀